# Kaggle runs for A2, A3, A8, A9, C2, C4 and cleaning rule 8

**Kaggle settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**. For the Hugging Face upload, add the secret `HF_TOKEN` (Add-ons > Secrets) and turn it on for this notebook.

**How to run a step:** set `STEP` in the next cell, then **Save Version > Save & Run All (Commit)**. You can close the browser. When it is done, open the version, go to **Output**, and download `to_commit.zip` (and `logits_*.zip` for Elif).

| STEP | When | What it does | About |
|---|---|---|---|
| `smoke` | any time | Runs all the code on a small fake dataset on the T4 (nothing to commit) | 30-40 min |
| `rule8` | now (the raw train audio is in the team repo) | NVIDIA CTC head on every train clip: `data/clean/rule8_cer.csv` + 50 flagged clips in `data/clean/manual_check.csv` | 30-60 min |
| `train_night` | after Gate 1 (clean data in the team repo) | C2 XLS-R on GPU 0 and A2 CTC on GPU 1 at the same time, then dev predictions and C2 dev logits | under 11.5 h |
| `train_day` | after `train_night` | A3 RNN-T on GPU 0, A8/A9 NVIDIA dev run on GPU 1 | under 11.5 h |
| `improve` | after `train_day` | C4: NVIDIA FastConformer fine-tuned on our train data (GPU 0) and C2 trained longer, "C2long" (GPU 1); then dev predictions. Starts with a 10-minute check on a few clips and a GPU memory test on the longest clips, then a learning-rate search for C4 (4 values, 1 epoch each, scored on dev). Training stops in time for Kaggle's 12-hour limit. | about 9 h |
| `test` | Saturday, after the freeze | USC test for A2, A3, A8, A9, C2, C2long and C4, C2 test logits | about 1.5 h |
| `upload` | any time after `train_night` | Uploads the C2 model to Hugging Face again (model card refresh) | 10 min |

For `test` and `upload`, first attach the outputs of the `train_night`, `train_day` and `improve` versions: **Add Input > Your Work > Notebooks > this notebook**, choose the version. The notebook finds the checkpoints by itself.

Test sets are used once: run `test` only after the freeze (plan B4d).

In [ ]:
# ---- the only lines you change ----
STEP = "smoke"          # smoke | rule8 | train_night | train_day | improve | test | upload
REPO_URL = "https://github.com/ata-sesli/nlp-project-3"         # team repo: the data comes from here
DATA_BRANCH = "main"    # branch of the team repo that has Ata's data/ folder
CODE_REPO_URL = "https://github.com/ata-sesli/nlp-project-3"   # repo with this code (the team repo)
CODE_BRANCH = "main"
HF_REPO = "Waxech/xls-r-300m-uzbek-usc"   # C2 model on Hugging Face ("" = no upload)
HF_REPO_C4 = "Waxech/stt-uz-fastconformer-usc"   # C4 model on Hugging Face ("" = no upload)
HOURS = 10.0            # training time per model; Kaggle stops a run at 12 h
IMPROVE_HOURS = 7.5     # training time for step improve (about 9 h in total); must fit in your GPU quota left
C4_LRS = [5e-5, 1e-4, 2e-4, 4e-4]   # learning rates tried for C4 (step improve); the best one on dev is used

In [ ]:
import csv, glob, json, math, os, shutil, subprocess, time, zipfile
from pathlib import Path

T_START = time.time()   # Kaggle stops a session after 12 hours
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)
REPO = Path("/tmp/repo")   # not in /kaggle/working, so the 2 GB of audio is not saved as output
SYSTEMS = ["A2_conformer-ctc", "A3_conformer-rnnt", "A8_nvidia-fastconformer-rnnt",
           "A9_nvidia-fastconformer-ctc", "C2_xlsr-300m-ctc", "C2long_xlsr-300m-ctc",
           "C4a_fastconformer-usc-ft-rnnt", "C4b_fastconformer-usc-ft-ctc", "C4_fastconformer-usc-ft"]
LONG = "C2long_xlsr-300m-ctc"   # C2 trained further (step improve); XLSR_SYSTEM gives it its own result files


def run(cmd, env=None):
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=REPO if REPO.exists() else None, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed: {cmd}")


def soft(cmd, env=None):
    """Like run(), but a failure is noted and the notebook goes on, so to_commit.zip is still made."""
    try:
        run(cmd, env=env)
        return True
    except RuntimeError as err:
        print(f"!! {err} (going on; listed again at the end)")
        problems.append(cmd)
        return False


def run_parallel(jobs, strict=True):
    """jobs = [(name, command)]; job k runs on GPU k. With one GPU they run one after the other.
    strict=False: returns the names of failed jobs instead of stopping."""
    import torch
    if torch.cuda.device_count() < len(jobs):
        print("Fewer GPUs than jobs: running one after the other (choose 'GPU T4 x2' to run them together).")
        if strict:
            for _, cmd in jobs:
                run(cmd)
            return []
        return [name for name, cmd in jobs if not soft(cmd)]
    procs = []
    for k, (name, cmd) in enumerate(jobs):
        log = WORK / f"log_{name}.txt"
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
        print(f"Starting {name} on GPU {k}, log: {log}")
        procs.append((name, log, subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, cwd=REPO, env=env)))
    while any(p.poll() is None for _, _, p in procs):
        time.sleep(600)
        for name, log, _ in procs:
            tail = log.read_text(errors="ignore").strip().splitlines()[-2:]
            print(f"[{time.strftime('%H:%M')}] {name}: {' | '.join(tail)[-300:]}", flush=True)
    for name, log, p in procs:
        print(f"\n===== {name} (exit code {p.returncode}) =====\n" + "\n".join(log.read_text(errors="ignore").splitlines()[-15:]))
    bad = [name for name, _, p in procs if p.returncode != 0]
    if bad and strict:
        raise RuntimeError(f"Failed: {bad}. See the log files in the output.")
    problems.extend(f"{name} (see log_{name}.txt in the output)" for name in bad)
    return bad


def pip_install(req):
    # Kaggle's own GPU builds of torch / torchaudio / numpy / scipy are kept.
    # numba-cuda is only installed when needed (see nemo_env)
    run(f"grep -vE '^(#|torch==|torchaudio==|numpy==|scipy==|numba-cuda)' {req} > /tmp/req.txt"
        " && pip install -q -r /tmp/req.txt")


NUMBA_GPU_TEST = """
import numpy as np
from numba import cuda

@cuda.jit
def add_one(a):
    a[0] += 1

x = cuda.to_device(np.zeros(1))
add_one[1, 1](x)
assert x.copy_to_host()[0] == 1
print("numba CUDA works")
"""


def numba_gpu_ok(py):
    """NeMo's RNN-T loss (training only) runs as numba CUDA kernels: compile and run a tiny one."""
    return subprocess.run([py, "-c", NUMBA_GPU_TEST], cwd="/tmp").returncode == 0


def nemo_env():
    """Installs NeMo (the same pinned packages as A8/A9) and checks that numba can run CUDA kernels, which
    NeMo's RNN-T loss needs for training. C2's packages are installed again after it (same versions, so
    nothing changes; this only makes sure), so C2 trains exactly as before. Returns the python to use."""
    pip_install("nvidia_finetune/requirements.txt")
    pip_install("xlsr_ctc/requirements.txt")
    if not numba_gpu_ok("python"):
        print("numba cannot run CUDA kernels yet: installing numba-cuda for CUDA 13 (NVIDIA's package for this)")
        run("grep -E '^numba-cuda' nvidia_finetune/requirements.txt > /tmp/req_numba.txt"
            " && pip install -q -r /tmp/req_numba.txt")
        if not numba_gpu_ok("python"):
            raise RuntimeError("numba still cannot run CUDA kernels, so NeMo cannot train the RNN-T head here.")
    return "python"


def find(pattern):
    """Newest match in the attached inputs (/kaggle/input) or in this run's output."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(f"{WORK}/{pattern}")
    hits.sort(key=os.path.getmtime)
    return hits[-1] if hits else None


def resume_arg(folder, marker):
    """' --resume-from <run folder>' when an attached earlier version has a checkpoint, else ''."""
    hits = glob.glob(f"/kaggle/input/**/{folder}/{marker}", recursive=True)
    if not hits:
        return ""
    newest = max(hits, key=os.path.getmtime)
    return " --resume-from " + newest[: newest.index(f"/{folder}/") + len(folder) + 1]


# ---- get the code (your repo) and the data/ folder (team repo)
shutil.rmtree(REPO, ignore_errors=True)
run(f"git clone --depth 1 -b {CODE_BRANCH} {CODE_REPO_URL} {REPO}")
if (REPO_URL, DATA_BRANCH) != (CODE_REPO_URL, CODE_BRANCH):
    fetched = subprocess.run(f"git fetch --depth 1 {REPO_URL} {DATA_BRANCH}", shell=True, cwd=REPO).returncode == 0
    has_data = fetched and subprocess.run("git ls-tree --name-only FETCH_HEAD data", shell=True, cwd=REPO,
                                          capture_output=True, text=True).stdout.strip()
    if has_data:
        run("git checkout FETCH_HEAD -- data")
    elif STEP == "smoke":
        print("The team repo has no data/ folder yet. That is fine for the smoke step.")
    else:
        raise SystemExit(f"No data/ folder in {REPO_URL} ({DATA_BRANCH}) yet. Only STEP = 'smoke' works without it.")
run("git log -1 --format='repo at %h %s' && nvidia-smi --query-gpu=name,memory.total --format=csv")
pip_install("xlsr_ctc/requirements.txt")
pip_install("baselines/scratch/requirements.txt")
NEMO_REQ = "baselines/nvidia_fastconformer/requirements.txt"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found.")
except Exception:
    print("No HF_TOKEN secret: the Hugging Face upload is skipped.")
to_commit = []   # repo files that go into to_commit.zip
problems = []    # steps that failed but did not stop the notebook

In [ ]:
XL, A2, A3 = WORK / "xlsr_run", WORK / "a2_ctc", WORK / "a3_rnnt"
C4, XL2 = WORK / "c4_run", WORK / "xlsr_long"
TEST = "--split test --allow-test"


def upload_c2(model_dir):
    if HF_REPO and os.environ.get("HF_TOKEN"):
        run(f"python xlsr_ctc/push_to_hub.py --model-dir {model_dir} --repo {HF_REPO} --code-url {REPO_URL}")


def zip_folder(folder, name):
    if Path(folder).exists():
        shutil.make_archive(str(WORK / name), "zip", folder)
        print(f"Wrote {WORK / name}.zip (send it to Elif for C3)")


if STEP == "smoke":
    FAKE = WORK / "smoke"
    os.environ["ASR_OUTPUT_ROOT"] = str(FAKE / "outputs")   # fake results never go into benchmark/
    C = f"--clean-dir {FAKE}/data/clean"
    run(f"python common/make_fake_data.py --out {FAKE}/data --train 300 --dev 40 --min-words 6 --max-words 12 --long 8")
    run(f"python xlsr_ctc/train.py --out-dir {FAKE}/xlsr {C} --max-steps 40 --warmup-steps 10 --eval-every 20 --save-every 20")
    run(f"python xlsr_ctc/predict.py --model {FAKE}/xlsr/best {C} --split dev --logits-dir {FAKE}/logits")
    run(f"python baselines/scratch/train.py --model ctc --out-dir {FAKE}/a2 {C} --epochs 2 --warmup-steps 10")
    run(f"python baselines/scratch/train.py --model rnnt --out-dir {FAKE}/a3 {C} --epochs 2 --warmup-steps 10")
    run(f"python baselines/scratch/predict.py --ckpt {FAKE}/a3/best.pt {C} --split dev")
    pip_install(NEMO_REQ)
    run(f"python baselines/nvidia_fastconformer/run_nvidia.py {C} --split dev --limit 20")
    run(f"python baselines/nvidia_fastconformer/rule8_flag.py --raw-dir {FAKE}/data/raw/usc/train --clean-dir {FAKE}/rule8 --limit 40")
    shutil.rmtree(FAKE / "xlsr", ignore_errors=True)   # 4 GB of test checkpoints, not needed
    print("\nSMOKE TEST PASSED. The line '... s/step' above is the XLS-R speed on this GPU.")

elif STEP == "rule8":
    pip_install(NEMO_REQ)
    run("python baselines/nvidia_fastconformer/rule8_flag.py --raw-dir data/raw/usc/train")
    to_commit += ["data/clean/rule8_cer.csv", "data/clean/manual_check.csv", NEMO_REQ]

elif STEP == "train_night":
    run_parallel([
        ("C2_xlsr", f"python xlsr_ctc/train.py --out-dir {XL} --max-hours {HOURS}" + resume_arg("xlsr_run", "last/trainer_state.json")),
        ("A2_ctc", f"python baselines/scratch/train.py --model ctc --out-dir {A2} --max-hours {HOURS}" + resume_arg("a2_ctc", "last.pt")),
    ])
    run(f"python xlsr_ctc/predict.py --model {XL}/best --split dev --logits-dir {WORK}/logits_dev")
    run(f"python baselines/scratch/predict.py --ckpt {A2}/best.pt --split dev")
    zip_folder(WORK / "logits_dev", "logits_dev_C2")
    shutil.copy(XL / "train_log.jsonl", REPO / "benchmark/results/C2_xlsr-300m-ctc_train_log.jsonl")
    upload_c2(XL / "best")
    to_commit += ["xlsr_ctc/requirements.txt", "baselines/scratch/requirements.txt"]

elif STEP == "train_day":
    pip_install(NEMO_REQ)
    run_parallel([
        ("A3_rnnt", f"python baselines/scratch/train.py --model rnnt --out-dir {A3} --max-hours {HOURS}" + resume_arg("a3_rnnt", "last.pt")),
        ("A8_A9_dev", "python baselines/nvidia_fastconformer/run_nvidia.py --split dev"),
    ])
    run(f"python baselines/scratch/predict.py --ckpt {A3}/best.pt --split dev")
    to_commit += ["baselines/scratch/requirements.txt", NEMO_REQ]

elif STEP == "improve":
    py = nemo_env()
    # 1) about 10 minutes on a few clips, so a mistake shows up now and not after 9 hours
    quick = dict(os.environ, ASR_OUTPUT_ROOT=str(WORK / "quick_check"), XLSR_SYSTEM=LONG)
    run(f"{py} nvidia_finetune/train.py --out-dir {WORK}/quick_c4 --limit-train 64 --val-limit 32 --epochs 1"
        " --warmup-steps 1 --max-hours 0.3", env=quick)
    run(f"{py} nvidia_finetune/predict.py --model {WORK}/quick_c4/best.nemo --split dev --limit 16", env=quick)
    run(f"{py} nvidia_finetune/push_to_hub.py --nemo {WORK}/quick_c4/best.nemo --repo you/check --dry-run", env=quick)
    # GPU memory: one step on the longest clips. If it does not fit, smaller pieces with the same total batch (32)
    C4_BATCH = None
    for bs, acc in ((16, 2), (8, 4), (4, 8)):
        cmd = (f"{py} nvidia_finetune/train.py --out-dir {WORK}/probe_c4 --memory-probe"
               f" --batch-size {bs} --grad-accum {acc}")
        print(f"\n$ {cmd}", flush=True)
        p = subprocess.run(cmd, shell=True, cwd=REPO, env=quick, capture_output=True, text=True)
        text = p.stdout + p.stderr
        if p.returncode == 0:
            print([l for l in text.splitlines() if l.startswith("MEMORY OK")][-1])
            C4_BATCH = (bs, acc)
            break
        if "out of memory" not in text.lower():
            print(text[-5000:])
            raise RuntimeError(f"Command failed: {cmd}")
        print(f"batch {bs} does not fit in GPU memory; trying {bs // 2}")
    if C4_BATCH is None:
        raise RuntimeError("Even batch 4 does not fit in GPU memory.")
    BS, ACC = C4_BATCH
    c2_quick = WORK / "quick_c2"
    run(f"python xlsr_ctc/train.py --out-dir {c2_quick} --model-id {HF_REPO} --limit-train 64 --limit-dev 16"
        " --lr 1e-4 --max-steps 10 --warmup-steps 2 --eval-every 10 --save-every 10", env=quick)
    c2_wer = [json.loads(l)["dev_wer"] for l in open(c2_quick / "train_log.jsonl") if '"dev_wer"' in l]
    if not c2_wer or c2_wer[-1] > 0.8:   # C2 gets about 0.34; near 1.0 means the start model was not read right
        raise RuntimeError(f"C2long check: dev WER {c2_wer} on 16 clips, expected about 0.3-0.5")
    for d in ("quick_c4", "probe_c4", "quick_c2", "quick_check"):
        shutil.rmtree(WORK / d, ignore_errors=True)
    print(f"QUICK CHECK PASSED. C4 batch {BS} x {ACC}; C2 start model dev WER on 16 clips: {c2_wer[-1]:.2f}")

    # 2) learning-rate search for C4: each value trains 1 epoch from the start model and is scored on the
    #    same 1,000 dev clips; two values at a time (one per GPU). The full run uses the best value.
    LR_DIR = WORK / "lr_search"
    for pair in (C4_LRS[:2], C4_LRS[2:]):
        run_parallel([(f"C4_lr_{lr:g}", f"ASR_OUTPUT_ROOT={LR_DIR} {py} nvidia_finetune/train.py"
                       f" --out-dir {LR_DIR}/lr_{lr:g} --lr {lr} --epochs 1 --warmup-steps 100"
                       f" --batch-size {BS} --grad-accum {ACC} --no-save --no-start-check") for lr in pair],
                     strict=False)
    search = {}
    for lr in C4_LRS:
        log = LR_DIR / f"lr_{lr:g}" / "train_log.jsonl"
        lines = log.read_text().splitlines() if log.exists() else []
        if lines:
            search[lr] = json.loads(lines[-1])
    if not search:
        raise RuntimeError("The learning-rate search failed for every value. See log_C4_lr_*.txt in the output.")
    best_lr = min(search, key=lambda lr: (round(search[lr]["val_wer"], 4), lr))
    # training time: IMPROVE_HOURS, but always leave 1 hour for predictions and saving before Kaggle's 12-hour stop
    hours_used = (time.time() - T_START) / 3600
    time_left = 11.3 - hours_used - 1.0
    if time_left < 1:
        raise RuntimeError("Not enough Kaggle time left for the real runs.")
    run_hours = round(min(IMPROVE_HOURS, time_left), 2)
    print(f"{hours_used:.1f} h used so far; the real runs get {run_hours} h")
    # epochs that fit in run_hours, from the measured speed (slowest trial, 10% spare)
    n_train = sum(1 for r in csv.DictReader(open(REPO / "data/clean/train.csv", encoding="utf-8"))
                  if float(r["end_s"] or r["duration_s"]) - float(r["start_s"] or 0) <= 30)
    sec_step = max(e.get("sec_per_step", 0) for e in search.values())
    val_min = max(e.get("val_minutes", 1) for e in search.values())
    epoch_min = (sec_step * math.ceil(math.ceil(n_train / BS) / ACC)) / 60 + val_min
    c4_epochs = max(5, min(30, int(run_hours * 60 * 0.9 / epoch_min))) if sec_step else 15
    table = {"method": "each learning rate trains 1 epoch from the start model (AdamW, warmup 100 steps, cosine);"
                       " WER of the transducer head on the same fixed sample of 1,000 dev clips (seed 42)",
             "results": [{"lr": lr, "dev_check_wer": round(e["val_wer"], 4),
                          "dev_check_wer_ctc": round(e.get("val_wer_ctc", float("nan")), 4),
                          "sec_per_step": e.get("sec_per_step")} for lr, e in sorted(search.items())],
             "chosen_lr": best_lr, "batch": f"{BS} x {ACC} accumulation steps", "full_run_epochs": c4_epochs,
             "minutes_per_epoch_estimate": round(epoch_min, 1)}
    (REPO / "benchmark/results").mkdir(parents=True, exist_ok=True)
    (REPO / "benchmark/results/C4_lr_search.json").write_text(json.dumps(table, indent=2) + "\n")
    print(json.dumps(table, indent=2))
    shutil.rmtree(LR_DIR, ignore_errors=True)

    # 3) the real runs, one per GPU; if one fails, the other one's results are still kept
    failed = run_parallel([
        ("C4_finetune", f"{py} nvidia_finetune/train.py --out-dir {C4} --lr {best_lr} --epochs {c4_epochs}"
                        f" --batch-size {BS} --grad-accum {ACC} --max-hours {run_hours}"),
        ("C2_long", f"XLSR_SYSTEM={LONG} python xlsr_ctc/train.py --out-dir {XL2} --model-id {HF_REPO}"
                    f" --lr 1e-4 --warmup-steps 300 --epochs 20 --max-hours {run_hours}"),
    ], strict=False)
    # 4) dev scores, logs, Hugging Face upload of C4
    if (C4 / "best.nemo").exists():
        soft(f"{py} nvidia_finetune/predict.py --model {C4}/best.nemo --split dev")
        if (C4 / "train_log.jsonl").exists():
            shutil.copy(C4 / "train_log.jsonl", REPO / "benchmark/results/C4_fastconformer-usc-ft_train_log.jsonl")
        shutil.rmtree(C4 / "ckpt", ignore_errors=True)   # best.nemo has the same weights
        if HF_REPO_C4 and os.environ.get("HF_TOKEN"):
            soft(f"{py} nvidia_finetune/push_to_hub.py --nemo {C4}/best.nemo --repo {HF_REPO_C4} --code-url {REPO_URL}")
        soft(f"{py} common/pin_requirements.py nvidia_finetune/requirements.txt")
        to_commit += ["nvidia_finetune/requirements.txt"]
    if (XL2 / "best" / "best.json").exists():
        soft(f"XLSR_SYSTEM={LONG} python xlsr_ctc/predict.py --model {XL2}/best --split dev"
             f" --logits-dir {WORK}/logits_dev_long")
        zip_folder(WORK / "logits_dev_long", "logits_dev_C2long")
        if (XL2 / "train_log.jsonl").exists():
            shutil.copy(XL2 / "train_log.jsonl", REPO / f"benchmark/results/{LONG}_train_log.jsonl")
        to_commit += ["xlsr_ctc/requirements.txt"]

elif STEP == "test":
    xl_best = find("xlsr_run/best/best.json")
    a2_best, a3_best = find("a2_ctc/best.pt"), find("a3_rnnt/best.pt")
    if not (xl_best or HF_REPO) or not a2_best or not a3_best:
        raise SystemExit("Attach the outputs of the train_night and train_day versions first (Add Input).")
    c2_model = str(Path(xl_best).parent) if xl_best else HF_REPO
    run(f"python xlsr_ctc/predict.py --model {c2_model} {TEST} --logits-dir {WORK}/logits_test")
    xl2_best, c4_best = find("xlsr_long/best/best.json"), find("c4_run/best.nemo")
    if xl2_best:
        run(f"XLSR_SYSTEM={LONG} python xlsr_ctc/predict.py --model {Path(xl2_best).parent} {TEST}"
            f" --logits-dir {WORK}/logits_test_long")
    run(f"python baselines/scratch/predict.py --ckpt {a2_best} {TEST}")
    run(f"python baselines/scratch/predict.py --ckpt {a3_best} {TEST}")
    pip_install(NEMO_REQ)
    rev_file = REPO / "benchmark/results/A8_nvidia-fastconformer-rnnt.json"
    rev = json.loads(rev_file.read_text()).get("revision") if rev_file.exists() else None
    run("python baselines/nvidia_fastconformer/run_nvidia.py " + TEST + (f" --revision {rev}" if rev else ""))
    if c4_best:
        run(f"python nvidia_finetune/predict.py --model {c4_best} {TEST}")
        if HF_REPO_C4 and os.environ.get("HF_TOKEN"):   # model card with the test scores too
            soft(f"python nvidia_finetune/push_to_hub.py --nemo {c4_best} --repo {HF_REPO_C4} --code-url {REPO_URL}")
    zip_folder(WORK / "logits_test_long", "logits_test_C2long")
    zip_folder(WORK / "logits_test", "logits_test_C2")
    if xl_best:
        upload_c2(Path(xl_best).parent)

elif STEP == "upload":
    xl_best = find("xlsr_run/best/best.json")
    if not xl_best:
        raise SystemExit("Attach the output of the train_night version first (Add Input).")
    upload_c2(Path(xl_best).parent)

else:
    raise SystemExit(f"Unknown STEP {STEP!r}")

In [ ]:
# ---- to_commit.zip: extract it inside your local repo folder, then commit and push
if STEP != "smoke":
    for req in ["xlsr_ctc/requirements.txt", "baselines/scratch/requirements.txt", NEMO_REQ]:
        if req in to_commit:
            run(f"python common/pin_requirements.py {req}")   # exact versions that ran here
    for system in SYSTEMS + ["C4_lr_search"]:
        to_commit += [str(p.relative_to(REPO)) for p in (REPO / "benchmark").glob(f"*/{system}*")]
    with zipfile.ZipFile(WORK / "to_commit.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for rel in sorted(set(to_commit)):
            if (REPO / rel).exists():
                z.write(REPO / rel, rel)
                print("added", rel)
    print(f"\nDone. Download {WORK / 'to_commit.zip'} from the Output tab.")
if problems:
    raise RuntimeError("These steps failed (the rest is saved in to_commit.zip):\n" + "\n".join(problems))